# Batch processing a folder with custom parameters

This example runs the pyKrait pipeline on every video in a folder using a single `AnalysisParameters` configuration that you define here.

Each video is processed independently and its results (CSVs, masks, projections, plots) are written to an `Analysis_<video_name>` subfolder next to the source file. After the run, a single overview CSV is created at the root of the folder.

In [ ]:
from pykrait.pipeline.pipeline import BatchExperiment, AnalysisParameters
from pykrait.io.files import concat_analysis_files

## 1. Configure the input folder

Set `folder` to the directory that contains your calcium-imaging videos. All files in this folder matching `extension` will be analyzed.

If you trained or downloaded a custom Cellpose model, point `cellpose_model_path` at the file. Use `"cpsam"` to fall back to the built-in Cellpose-SAM model.

In [ ]:
folder = "/path/to/your/videos"
extension = ".czi"  # also supports ".tif" / ".tiff"
cellpose_model_path = "cpsam"  # or "/path/to/custom_cellpose_model"

## 2. Define the analysis parameters

Every field of `AnalysisParameters` is optional and falls back to a sensible default. Times are given in seconds and distances in micrometers; pyKrait converts to frames/pixels internally using the metadata of each video.

In [ ]:
params = AnalysisParameters(
    # Segmentation
    tproj_type="std",            # "std" or "sum" projection across time
    CLAHE_normalize=True,        # contrast-enhance the projection before Cellpose
    cellpose_model_path=cellpose_model_path,
    frame_interval=None,         # seconds, if None, it will try to read it from the metadata of the video, if not found, it will default to 1.0 seconds
    # Detrending
    sinc_filter_window=300,      # cutoff period for the sinc filter in seconds, usually half of the acquisition time
    # Peak detection
    peak_min_width=1.0,          # seconds
    peak_max_width=80.0,         # seconds
    peak_prominence=1500,        # artificial units, see https://docs.scipy.org/doc/scipy/reference/generated/scipy.signal.peak_prominences.html
    peak_min_height=800,         # artificial units, minimum value of a peak
    # Periodicity
    periodicity_method="cutoff", # "cutoff" or "quantile", if "cutoff", it will use the std_threshold and cov_threshold to identify periodic cells, if "quantile", it will use the quantiles of the experimental distribution of std and cov to identify periodic cells based on the periodicity_quantiles parameter
    std_threshold=15.0,          # if "cutoff", standard deviation of peak-peak interval in seconds, if "quantile" the quantile (must be between 0 and 1)
    cov_threshold=0.2,           # if "cutoff", coefficient of variation of peak-peak interval, if "quantile" the quantile (must be between 0 and 1)
    # Synchronicity
    time_window=10.0,            # Time in seconds in which the peaks of neighbouring cells are still considered synchronous
    neighbour_tolerance=10.0,    # pixels, maximum distance between two cells to be considered neighbours for the synchronicity analysis
)

## 3. Run the batch

`BatchExperiment(..., load_params=False)` applies the same `params` object to every video. Set `skip_cellpose=True` if cell masks already exist on disk and you want to re-use them; set `skip_intensity_extraction=True` to also re-use cached `*_raw_intensities.csv.zst` files.

In [ ]:
experiment = BatchExperiment(
    folder=folder,
    params=params,
    extension=extension,
    load_params=False,
)

experiment.run(
    skip_cellpose=False,
    skip_intensity_extraction=False,
)

## 4. Generate an overview CSV

`concat_analysis_files` walks the folder tree and merges all per-video result files into a single overview CSV at the root of the folder.

In [ ]:
concat_analysis_files(folder, filetype="output")
concat_analysis_files(folder, filetype="parameters")